In [ ]:
# --- Colab bootstrap: in Colab, clone + install the repo; locally, do nothing ---
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/CKmellow/per3-genotype-clock-gene-timing.git"
BRANCH = "main"  # change to your branch name to test it in Colab
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    repo = Path("/content/per3-genotype-clock-gene-timing")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(repo)], check=True)
    os.chdir(repo)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
    src = str(repo / "src")
else:  # local: find the repo root (works from notebooks/ or notebooks/stages/)
    here = Path.cwd().resolve()
    root = next((p for p in [here, *here.parents] if (p / "pyproject.toml").exists()), here)
    src = str(root / "src")
if src not in sys.path:
    sys.path.insert(0, src)  # makes `import per3clock` work without a kernel restart

from per3clock import config  # noqa: E402

config.set_seeds()
config.ensure_dirs()
print("per3clock root:", config.ROOT, "| Colab:", IN_COLAB)

# Stage 1 · Research question and objectives

**Research question.** Does PER3 genotype (4/4 vs 5/5) shift the time of day at which the clock genes PER2, PER3 and NR1D1 reach peak expression in human blood?

**Hypothesis.** PER3 5/5 ("early bird") carriers reach peak clock-gene expression earlier in the day than 4/4 ("night owl") carriers.

**Objectives**
1. To estimate the time of peak expression of PER2, PER3 and NR1D1 in each participant.
2. To compare peak timing between the PER3 4/4 and 5/5 genotype groups and estimate the size of any shift.
3. To determine whether any shift reflects a change in the whole internal clock, using melatonin-based circadian phase.
4. To determine whether the results support the hypothesis.

## Background

**PER3 length polymorphism and chronotype.** *PER3* (Period 3) carries a variable-number tandem repeat (VNTR) present as 4 or 5 copies, giving 4/4, 4/5 and 5/5 genotypes. Archer et al. (2003) found that the polymorphism correlated significantly with extreme diurnal preference: *"the longer allele associating with morningness and the shorter allele with eveningness"*. The shorter allele was also strongly associated with delayed sleep phase syndrome. This is why 5/5 carriers are called "early birds" and 4/4 carriers "night owls".

**The molecular clock.** In each cell, CLOCK and ARNTL (BMAL1) drive transcription of the *PER* and *CRY* repressors and of *NR1D1/NR1D2* (REV-ERBα/β). These feed back on the loop and give ~24 h oscillations. Peripheral blood cells keep such rhythms, so the timing of peak expression (the **acrophase**) of PER2, PER3 and NR1D1 can be read from repeated blood samples.

**Dataset.** Möller-Levet et al. (2013) collected 10 whole-blood RNA samples from each of 26 participants during ~40 h of total sleep deprivation. This followed one week of sufficient sleep (control, ~8.5 h) and, in a crossover, one week of insufficient sleep (~5.7 h). They reported 1,855 genes with a circadian expression profile in the control condition, and listed PER1–3, CLOCK, NR1D1 and NR1D2 among the circadian-related genes affected by insufficient sleep. Participants were homozygous for the PER3 VNTR (12 with the shorter allele). The paper does not analyse peak timing by PER3 genotype, which is the gap this project addresses. Each sample also carries a **melatonin-based circadian phase**, so we can separate a shift in clock-gene timing from a shift in the whole internal clock (objective 3).

**Design choices.** The main analysis uses the sufficient-sleep ("Sleep Extension") condition. "Sleep Restriction" is used only as a sensitivity analysis. Primary genes are PER2, PER3 and NR1D1. CLOCK, CRY1, ARNTL, PER1 and NR1D2 are secondary.

In [ ]:
print("Primary genes:  ", ", ".join(config.GENES_PRIMARY))
print("Secondary genes:", ", ".join(config.GENES_SECONDARY))
print("Main condition: ", config.MAIN_CONDITION, "| sensitivity:", config.SENSITIVITY_CONDITION)
print("Period (h):     ", config.PERIOD_H, "| seed:", config.SEED)

## References
- Archer SN, Robilliard DL, Skene DJ, Smits M, Williams A, Arendt J, von Schantz M (2003). A length polymorphism in the circadian clock gene *Per3* is linked to delayed sleep phase syndrome and extreme diurnal preference. *Sleep* 26(4):413–415. doi:10.1093/sleep/26.4.413. PMID 12841365.
- Möller-Levet CS, Archer SN, Bucca G, et al. (2013). Effects of insufficient sleep on circadian rhythmicity and expression amplitude of the human blood transcriptome. *PNAS* 110(12):E1132–E1141. doi:10.1073/pnas.1217154110. PMID 23440187.
- GEO accession GSE39445. https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE39445

### Interpretation
The question is specific (three named genes, one genetic contrast, one outcome: peak time), biological, and testable with public data. GSE39445 has repeated blood samples across the 24-h cycle in genotyped participants, plus melatonin-based phase. Objectives 1–2 test the hypothesis directly. Objective 3 asks whether any difference reflects the whole internal clock (a shift relative to clock time that disappears relative to melatonin phase) or clock-gene timing specifically. Objective 4 brings the evidence together.